# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing data elements by their Croissant `@id` fields for clarity and reliability.

### Dataset Source
The dataset Croissant schema is available at:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` (and plotting libraries) are installed
!pip install -q mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This gives us programmatic access to all RecordSet, Field, and Column definitions via their unique `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an mlcroissant.Metadata object

# View dataset metadata summary
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The following cell will list all record sets available in this dataset, along with the fields (columns) for each, referencing their `@id`.

> **Note:** All references use the Croissant `@id` for reliable identification.

In [ ]:
# Fetch all record sets in the dataset
record_sets = dataset.record_sets
print("Available Record Sets (by @id and name):\n")
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")
# We'll print the fields for each record set
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs.id} (name: {rs.name})")
    print("Fields (@id, name, dataType):")
    for field in rs.fields:
        print(f"  - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

Let's inspect a **sample of records** from the primary `protein_records` record set, using the Croissant `@id`.

- We'll choose a main record set, such as '@id': `protein_records`, for exploration. 
- Entities such as record sets, fields, and columns are always referenced by their Croissant `@id`.

In [ ]:
# Select a record set to examine records; update this value to explore others as needed
# Common @id pattern: The main protein data is typically in a set named like 'cr:proteinRecords' or similar

# Find likely data-bearing record sets for inspection
if len(record_sets) > 0:
    main_record_set_id = record_sets[0].id  # For demonstration, take the first record set
else:
    raise RuntimeError("No record sets found in the dataset.")
print(f"\nListing 3 example records from RecordSet with @id: {main_record_set_id}\n")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(json.dumps(record, indent=2))
    if i >= 2:  # Show only the first 3 records
        break

## 3. Data Extraction
Load data from record sets into Pandas DataFrames for analysis. Use only Croissant `@id` for referencing the sets and fields.

We'll demonstrate extracting the first data-bearing record set (choose another by changing `main_record_set_id`).

In [ ]:
# Extract data from available record sets
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df

print(f"\nColumns (by Croissant field name) in DataFrame for '{main_record_set_id}':\n")
print(dataframes[main_record_set_id].columns.tolist())

print("\nFirst five records of the DataFrame:")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Now let's perform some basic EDA using fields referenced by their Croissant `@id`/column names as in the metadata.

Examples: filtering, normalization, and grouping on a numeric field such as `molecular_weight` or `normalized_abundance`. **Inspect the column names above and adjust field references as needed for your analysis.**

In [ ]:
# For demonstration, let's try a numeric field typical in proteomics: 'molecular_weight' or similar
# Update 'molecular_weight' and 'protein_class' below to actual column names from your exploration
numeric_field = None
candidate_numeric_fields = ['molecular_weight', 'Normalized_Abundance', 'molecular_weight_(MW)', 'coverage_percent', 'peptide_count']
df = dataframes[main_record_set_id]
for cand in candidate_numeric_fields:
    if cand in df.columns:
        numeric_field = cand
        break
if not numeric_field:
    raise RuntimeError("No suitable numeric field found. Please inspect the column names and set a numeric field.")
print(f"Using numeric field: {numeric_field}")
threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as threshold for demonstration
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.3f} (75th percentile): {len(filtered_df)} records")

# Normalize
filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"Normalized {numeric_field} for filtered records (sample):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt grouping by a categorical field
candidate_group_fields = ['protein_class', 'description', 'accession', 'protein_family']
group_field = None
for cand in candidate_group_fields:
    if cand in filtered_df.columns:
        group_field = cand
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by '{group_field}':")
    display(grouped_df.head())
else:
    print("No suitable grouping field found in DataFrame.")

## 5. Visualization
Visualize the distribution of the chosen numeric field, and optionally a boxplot by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group if available
if group_field:
    plt.figure(figsize=(10,4))
    order = df[group_field].value_counts().index[:10]  # Show top 10 by frequency
    sns.boxplot(data=df, x=group_field, y=numeric_field, order=order)
    plt.title(f"{numeric_field} by {group_field} (Top 10 categories)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We've loaded a complex proteomics FAIR² dataset using the Croissant schema with `mlcroissant`, explored schema-driven record sets and fields by their unique `@id`, and performed basic EDA and visualization. For specific downstream analyses or to process additional record sets, repeat the above steps, always referencing the appropriate Croissant `@id` for all entity selections.